Before Cleaning, we ensure that the data is available at any moment 

In [0]:
from pyspark.sql import functions as F

spark.conf.set("spark.sql.session.timeZone", "Europe/Brussels")

bronze = spark.table("workspace.energy.bronze_public_power")

bronze.printSchema()

null_counts = bronze.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in bronze.columns ])

display(null_counts)

In [0]:
%sql
SELECT *
FROM energy.bronze_public_power
LIMIT 20;

In [0]:
%sql
SELECT DATE_TRUNC('month', timestamp) AS month,  -- we get the month
COUNT(*) as missing_hours                        -- we get the count
FROM workspace.energy.bronze_public_power
WHERE nuclear is NULL      
GROUP BY DATE_TRUNC('month', timestamp)
ORDER BY 2

In [0]:
%sql
SET TIME ZONE 'Europe/Brussels';


SELECT DATE_TRUNC('day', timestamp) AS month,  -- we get the month
COUNT(*) as missing_hours                        -- we get the count
FROM workspace.energy.bronze_public_power
WHERE nuclear is NULL      
AND YEAR(`timestamp`) = 2026 AND MONTH(`timestamp`) IN (4, 5, 6, 7, 8, 9, 10)
GROUP BY DATE_TRUNC('day', timestamp)
ORDER BY 1

In [0]:
%sql
SET TIME ZONE 'Europe/Brussels';

SELECT DATE_TRUNC('month', `timestamp`) AS month,
       ROUND(AVG(nuclear), 1) AS avg_mw,        -- 
       ROUND(MIN(nuclear), 1) AS min_mw,
       ROUND(MAX(nuclear), 1) AS max_mw,
       COUNT(nuclear)         AS hours_with_data
FROM workspace.energy.bronze_public_power
WHERE `timestamp` >= '2026-01-01'
GROUP BY 1
ORDER BY 1;

In [0]:
from pyspark.sql import functions as F

# Read bronze
bronze = spark.table("workspace.energy.bronze_public_power")

# Source -> category mapping
categories = {
    "nuclear": "nuclear",
    "fossil_gas": "fossil",
    "fossil_oil": "fossil",
    "solar": "renewable",
    "wind_onshore": "renewable",
    "wind_offshore": "renewable",
    "hydro_run_of_river": "renewable",
    "biomass": "renewable",
    "waste": "other",
    "others": "other",
    "hydro_pumped_storage": "storage",
    "hydro_pumped_storage_consumption": "storage",
    "battery": "storage",
    "cross_border_electricity_trading": "exchange",
}
source_columns = list(categories.keys())

# Cast every source column to double (unpivot requires a single type)
for c in source_columns:
    bronze = bronze.withColumn(c, F.col(c).cast("double"))

outage_gap = (
    F.col("nuclear").isNull()
    & (F.col("timestamp") >= "2026-05-01")
    & (F.col("timestamp") < "2026-10-01")
)
bronze = bronze.withColumn("nuclear_imputed", outage_gap)
bronze = bronze.withColumn("nuclear", F.when(outage_gap, F.lit(0.0)).otherwise(F.col("nuclear")))

# Wide -> long: one row per (timestamp, source)
silver = bronze.unpivot(
    ids=["timestamp", "nuclear_imputed"],   
    values=source_columns,                      # which column to read the value from
    variableColumnName="source",                # name of the column who receive the column NAMES
    valueColumnName="mw",                       # name of the column who receive the column VALUES5
)



# Keep the imputed flag only on nuclear rows (unpivot copied it onto every source)
silver = silver.withColumn(
    "is_imputed",
    (F.col("source") == "nuclear") & F.col("nuclear_imputed"),
)

# Build the category lookup table as a Spark DataFrame
mapping_rows = []
for source, category in categories.items():
    mapping_rows.append((source, category))

mapping_df = spark.createDataFrame(mapping_rows, ["source", "category"])

# Add the category column (left join: keep every silver row, even if a source is missing from the mapping)
silver = silver.join(mapping_df, on="source", how="left")

# Remaining nulls (battery before 2025) = no data -> drop
silver = silver.filter(F.col("mw").isNotNull())

# Final columns
silver = silver.select("timestamp", "source", "category", "mw", "is_imputed")

# Write the silver Delta table
silver.write.format("delta").mode("overwrite").saveAsTable("workspace.energy.silver_generation")
print("Silver table saved ✅")

In [0]:
%sql
SELECT COUNT(*)                     AS nb_rows,
       SUM(CAST(is_imputed AS INT)) AS imputed_rows,
       MAX(`timestamp`)             AS last_hour
FROM workspace.energy.silver_generation;

In [0]:
%sql
SELECT source, COUNT(*) AS nb_rows
FROM workspace.energy.silver_generation
GROUP BY source
ORDER BY nb_rows;

In [0]:
%sql
SET TIME ZONE 'Europe/Brussels';

SELECT DATE_TRUNC('day', `timestamp`) AS day,
       COUNT(*) AS null_hours
FROM workspace.energy.bronze_public_power
WHERE fossil_oil IS NULL
GROUP BY 1
ORDER BY 1;

In [0]:
display(silver.orderBy("timestamp", "source").limit(14))

In [0]:
%sql
SELECT *
FROM workspace.energy.silver_generation
